In [ ]:
import torch
import gym
import numpy as np
import pandas as pd
import torch.nn as nn
from gym import spaces
import geopandas as gpd
import torch.optim as optim
from tabulate import tabulate
from geopy.geocoders import Nominatim
from sklearn.preprocessing import LabelEncoder
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
gpkg_filepath = "1KmIRLJoinedDataset.geojson"
gdf = gpd.read_file(gpkg_filepath)
print(gdf.head())

In [ ]:
required_columns = ["ENG_NAME_VALUE", "ED_Area (m^2)", "Count_EVCSInED",	"ED_area(km^2)",	"All usual residents", "All household spaces", "Whole House Or Bungalow",	"Flat, maisonette or appartment including bedsit NI	Caravan or other mobile or temporary structure",	"1 person in household",	"2 people in household", "3 people in household", "4 people in household",	"5 people in household", "6 people in household",	"7 people in household", "8 or more people in household",	"No cars or vans in household", "1 car or van in household", "2 cars or vans in household",	"3 cars or vans in household",	"4 or more cars or vans in household",	"All cars or vans",	"All Students	At Work", "Economically active: Unemployed: Aged 16-74 years",	"Economically active: Full-time student: Aged 16-74 years", "Economically inactive: Retired: Aged 16-74 years",	"Economically inactive: Looking after home or family: Aged 16-74 years",	"Economically inactive: Long-term sick or disabled: Aged 16-74 years",	"Count_TransHubsInED", "Count_EmerServInED",	"Count_CivicInED", "Count_PofWInED",	"Count_EducInED",	"Count_HealthcareInED",	"Count_AccomInED",	"LengthMWPInED",	"RoadLengthInED", "Count_FuelAreasInED", "Count_MSUNDGInED",	"Count_InterchangesInED",	"Count_IntersectionsInED",	"Count_MallsInED",	"Count_ParkingAreasInED", "Count_SubStatInED", "LengthPowerLinesInED",	"name", "socket_typ", "socket_cha", "socket_t_1", "socket_tes", "socket_sch",	"socket_t_2",	"socket_t_3",	"socket_t_4",	"amenity_wi",	"brand",	"socket_com", "socket_c_1",	"charge	voltage",	"amperage",	"socket_t_6",	"socket_c_2",	"socket_t_7",	"ref",	"operator_1",	"operator_2", "operator	socket_cee", "EVCS",	"Distance_to_accom_POI (m)",	"Distance_to_fuel_area (m)",	"Distance_to_green_area (m)",	"Distance_to_malls_or_departments_POI (m)",	"Distance_to_parking_area_any (m)",	"Distance_to_civic_POI (m)", "Distance_to_education_POI (m)",	"Distance_to_Emerg_Servs_POI (m)",	"Distance_to_Healthcare_POI (m)",	"Distance_to_major_water_area_POI (m)",	"Distance_to_major_water_path_POI (m)",	"Distance_to_major_water_path/path_POI (m)",	"Distance_to_MS_UNDG_&_paid_parking",	"Distance_to_PlaceOfWorship_POI (m)",	"Distance_to_Power_Lines (m)",	"Distance_to_Road_intersection (m)",	"Distance_to_Main_Roads (m)",	"Distance_to_Substation (m)", "Distance_to_Transport_Hub (m)",	"Distance_to_Motorway_Access/Egress (m)", "Slope (degrees)1",	"Elevation (m)1"]
gdf = gdf[required_columns] if set(required_columns).issubset(gdf.columns) else gdf.reindex(columns=required_columns, fill_value=0)

categorical_columns = ["ENG_NAME_VALUE", "Energy_Avail", "Energy_Consum", "socket_typ", "brand", "operator_1"]
numeric_columns = [col for col in required_columns if col not in categorical_columns]

gdf[numeric_columns] = gdf[numeric_columns].apply(pd.to_numeric, errors='coerce')
gdf[numeric_columns] = gdf[numeric_columns].fillna(0)
gdf[numeric_columns] = gdf[numeric_columns].replace([np.inf, -np.inf], 0)

label_encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    gdf[col] = le.fit_transform(gdf[col])
    label_encoders[col] = le

categorical_tensor = torch.tensor(gdf[categorical_columns].values, dtype=torch.long)
embedding_sizes = [(len(label_encoders[col].classes_), min(50, (len(label_encoders[col].classes_) // 2))) for col in categorical_columns]
embedding_layers = [torch.nn.Embedding(num_categories, embedding_dim) for num_categories, embedding_dim in embedding_sizes]
embedded_features = [embedding_layers[i](categorical_tensor[:, i]) for i in range(len(embedding_layers))]
categorical_embeddings = torch.cat(embedded_features, dim=1)
numerical_tensor = torch.tensor(gdf[numeric_columns].values, dtype=torch.float32)
state_data = torch.cat([numerical_tensor, categorical_embeddings], dim=1)

print("\nFormatted Data Preview:")
print(tabulate(gdf.head(10), headers='keys', tablefmt='grid'))
print("Final state_data shape:", state_data.shape)

In [ ]:
class EVChargingEnv(gym.Env):
    def __init__(self, state_data, gdf):
        super(EVChargingEnv, self).__init__()

        if isinstance(state_data, torch.Tensor):
            self.state_data = state_data.detach().cpu().numpy()
        elif not isinstance(state_data, np.ndarray):
            state_data = np.array(state_data, dtype=np.float32)

        self.gdf = gdf
        self.num_nodes = self.state_data.shape[0]

        self.action_space = spaces.MultiBinary(self.num_nodes)

        self.observation_space = spaces.Box(
            low=0, high=1, shape=(self.num_nodes, self.state_data.shape[1]), dtype=np.float32
        )

    def step(self, action):
        if isinstance(action, torch.Tensor):
            action = action.detach().cpu().numpy().astype(np.float32)

        action = np.asarray(action).flatten()

        if action.shape[0] != self.num_nodes:
            raise ValueError(f"Action shape mismatch: Expected ({self.num_nodes},) but got {action.shape}")

        if isinstance(self.state_data, torch.Tensor):
            self.state_data = self.state_data.detach().cpu().numpy()

        node_weights = np.sum(self.state_data * action[:, np.newaxis], axis=0)

        travel_cost = np.sum(action * self.state_data[:, 0])
        grid_cost = np.sum(action * self.state_data[:, 2])

        node_weight_sum = np.sum(node_weights)
        max_node_weight = np.max(node_weights) if np.max(node_weights) > 0 else 1
        normalized_reward = node_weight_sum / (max_node_weight + 1e-8)

        travel_cost_scaled = travel_cost / (np.max(self.state_data[:, 0]) + 1e-8)
        grid_cost_scaled = grid_cost / (np.max(self.state_data[:, 2]) + 1e-8)

        reward = normalized_reward - 0.1 * (travel_cost_scaled + grid_cost_scaled) + 1.0

        done = True

        return self.state_data, reward, done, {}

    def reset(self):
        return self.state_data

    def render(self):
        pass

def train(env, model, optimizer, num_episodes=10000):
    for episode in range(num_episodes):
        state = env.reset()
        state = torch.FloatTensor(state)

        log_probs = []
        values = []
        rewards = []

        action_probs, value = model(state)
        action_probs = action_probs.squeeze()

        if action_probs.ndim == 2 and action_probs.shape[1] != 1:
            action_probs = action_probs[:, 0]

        action = (action_probs > 0.5).float()
        log_prob = torch.log(action_probs + 1e-8)
        log_probs.append(log_prob)
        values.append(value)

        action_np = action.detach().cpu().numpy().astype(np.float32).flatten()

        if action_np.shape[0] != env.num_nodes:
            raise ValueError(f"Action shape mismatch: Expected ({env.num_nodes},) but got {action_np.shape}")

        _, reward, _, _ = env.step(action_np)
        reward = np.clip(reward, -10, 10)
        rewards.append(reward)

        advantage = (reward - value.detach().cpu().numpy().flatten()).mean()
        advantage = torch.tensor(advantage, dtype=torch.float32, device=value.device)

        reward_tensor = torch.full_like(value.squeeze(), reward, dtype=torch.float32)

        policy_loss = (-log_prob.mean() * advantage).mean()
        value_loss = nn.MSELoss()(value.squeeze(), reward_tensor).mean()

        loss = policy_loss + 0.5 * value_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if episode % 100 == 0:
            print(f"Episode {episode}: Reward {reward}")

class ActorCritic(nn.Module):
    def __init__(self, state_size, action_size):
        super(ActorCritic, self).__init__()

        self.shared = nn.Sequential(
            nn.Linear(state_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.actor = nn.Sequential(
            nn.Linear(64, action_size),
            nn.Sigmoid()
        )

        self.critic = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        shared_out = self.shared(state)
        action_probs = self.actor(shared_out)
        action_probs = torch.clamp(action_probs, min=0.1, max=0.9)
        value = self.critic(shared_out)
        return action_probs.squeeze(dim=1), value

env = EVChargingEnv(state_data, gdf)

state_size = state_data.shape[1]
action_size = state_data.shape[0]

model = ActorCritic(state_size, action_size)
optimizer = optim.Adam(model.parameters(), lr=0.01)

train(env, model, optimizer)

def save_results_to_gpkg(gdf, actions, output_filepath):
    if actions.ndim == 2 and actions.shape[1] != 1:
            actions = actions[:, 0]

    actions = actions.flatten()

    if actions.shape[0] != gdf.shape[0]:
        raise ValueError(f"Shape mismatch: gdf has {gdf.shape[0]} rows, but actions has {actions.shape[0]} elements")

    gdf["EVCS_Prediction"] = actions.numpy().astype(int)
    gdf.to_file(output_filepath, driver="GPKG")

state = torch.FloatTensor(state_data)
action_probs, _ = model(state)

actions = (action_probs > 0.5).float()

if not isinstance(gdf, gpd.GeoDataFrame):
    gdf = gpd.GeoDataFrame(gdf)

output_gpkg_filepath = "evcs_predictions.gpkg"
save_results_to_gpkg(gdf, actions, output_gpkg_filepath)


In [ ]:
def update_existing_gpkg(gpkg_filepath, encoded_actions, label_encoders):
    gdf = gpd.read_file(gpkg_filepath)
    print(f"Loaded GPKG with {gdf.shape[0]} records")

    encoded_actions = encoded_actions[:, 0].detach().cpu().numpy().astype(int).flatten()
    print(f"DEBUG: Encoded actions shape: {encoded_actions.shape}")

    if encoded_actions.shape[0] != gdf.shape[0]:
        raise ValueError(f"Shape mismatch: gdf has {gdf.shape[0]} rows, but actions have {encoded_actions.shape[0]} elements.")

    for col in label_encoders.keys():
        if col in gdf.columns:
            le = label_encoders[col]
            gdf[col] = le.inverse_transform(gdf[col].astype(int))

    gdf["EVCS_Prediction"] = encoded_actions

    if not isinstance(gdf, gpd.GeoDataFrame):
        gdf = gpd.GeoDataFrame(gdf)

    gdf.to_file(gpkg_filepath, driver="GPKG")
    print(f"Updated results saved to {gpkg_filepath}")

    return gdf

update_existing_gpkg("evcs_predictions.gpkg", actions, label_encoders)


In [ ]:
gdf_predicted = gdf[gdf["EVCS_Prediction"] == 1].copy()
gdf_predicted["centroid"] = gdf_predicted.geometry.centroid

geolocator = Nominatim(user_agent="evcs_locator")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1, error_wait_seconds=5.0)

evcs_rows = []

for idx, row in gdf_predicted.iterrows():
    eng_name = row["ENG_NAME_VALUE"]
    point = row["centroid"]

    try:
        location = reverse((point.y, point.x), language="en", addressdetails=True)
        county = location.raw["address"].get("county", "Unknown")
    except Exception as e:
        county = "Unknown"

    evcs_rows.append({
        "Territory": "RoI",
        "County": county,
        "Latitude": point.y,
        "Longitude": point.x
    })

evcs_df = pd.DataFrame(evcs_rows, columns=["Territory", "County", "Latitude", "Longitude"])
print(evcs_df)